# 4. Evaluate and compare all decoders

This notebook loads the saved test split and saved model checkpoints, evaluates correction-label and component accuracies, and saves a comparison table plus plots. The MLP, lookup, and QNN use only the latest observed round; the GRU uses all three rounds.

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
sys.path.insert(0, str(Path.cwd().resolve()))
from main import (load_split, MLPDecoder, GRUDecoder, predict_labels, lookup_predict,
                  evaluate_predictions, batch_loss, save_json)

device = "cuda" if torch.cuda.is_available() else "cpu"
test = load_split("data/test.npz")
y_test = np.stack([test["labels_x"], test["labels_z"]], axis=1)
x_mlp = test["observed"][:, -1, :].astype(np.float32)
x_gru = test["observed"].astype(np.float32)
results = {}
results["lookup"] = evaluate_predictions(lookup_predict(test["observed"]), y_test)

mlp = MLPDecoder().to(device)
mlp_ckpt = torch.load("models/mlp.pt", map_location=device, weights_only=False)
mlp.load_state_dict(mlp_ckpt["model_state_dict"])
results["mlp"] = evaluate_predictions(predict_labels(mlp, x_mlp, device=device), y_test)

gru = GRUDecoder(hidden_size=24).to(device)
gru_ckpt = torch.load("models/gru.pt", map_location=device, weights_only=False)
gru.load_state_dict(gru_ckpt["model_state_dict"])
results["gru"] = evaluate_predictions(predict_labels(gru, x_gru, device=device), y_test)

## Rebuild and load the QNN

Reconstruction must match the circuit used for training: same parameter order, observables, layer count, and classical head. The statevector estimator is local and exact.

In [ ]:
N_QUBITS, N_LAYERS = 6, 2
input_params = ParameterVector("x", N_QUBITS)
weight_params = ParameterVector("theta", N_LAYERS * N_QUBITS * 2)
qc = QuantumCircuit(N_QUBITS)
for qubit in range(N_QUBITS):
    qc.ry(np.pi * input_params[qubit], qubit)
cursor = 0
for layer in range(N_LAYERS):
    for qubit in range(N_QUBITS):
        qc.ry(weight_params[cursor], qubit); cursor += 1
        qc.rz(weight_params[cursor], qubit); cursor += 1
    for qubit in range(N_QUBITS - 1):
        qc.rzz(0.5, qubit, qubit + 1)

def pauli_string(occupied_qubits, n_qubits=6):
    chars = ["I"] * n_qubits
    for qubit, pauli in occupied_qubits:
        chars[n_qubits - 1 - qubit] = pauli
    return "".join(chars)
observables = [SparsePauliOp(pauli_string([(q, "Z")])) for q in range(N_QUBITS)]
observables += [SparsePauliOp(pauli_string([(q, "Z"), (q+1, "Z")]))
                for q in range(N_QUBITS-1)]
qnn = EstimatorQNN(circuit=qc, estimator=StatevectorEstimator(),
                   input_params=list(input_params), weight_params=list(weight_params),
                   observables=observables, input_gradients=True)
quantum_layer = TorchConnector(qnn, initial_weights=np.zeros(len(weight_params)))

class HybridQNN(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.quantum_layer = layer
        self.head = nn.Linear(len(observables), 16)
    def forward(self, x):
        return self.head(self.quantum_layer(x)).view(-1, 2, 8)

qnn_model = HybridQNN(quantum_layer).to(device)
qnn_ckpt = torch.load("models/qnn.pt", map_location=device, weights_only=False)
qnn_model.load_state_dict(qnn_ckpt["model_state_dict"])
qnn_model.eval()
qnn_logits = []
started = time.perf_counter()
with torch.no_grad():
    for start in range(0, len(x_mlp), 16):
        xb = torch.tensor(x_mlp[start:start+16], dtype=torch.float32, device=device)
        qnn_logits.append(qnn_model(xb).cpu().numpy())
qnn_inference_seconds = time.perf_counter() - started
qnn_logits = np.concatenate(qnn_logits)
results["qnn"] = evaluate_predictions(qnn_logits, y_test)
print("QNN test metrics:", results["qnn"])

In [ ]:
rows = []
for name, metrics in results.items():
    rows.append({"model": name, **metrics})
comparison = pd.DataFrame(rows)
display(comparison)
Path("results").mkdir(exist_ok=True)
comparison.to_csv("results/test_comparison.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(comparison["model"], comparison["correction_label_accuracy"])
plt.ylabel("Exact two-component label accuracy")
plt.title("Test accuracy by decoder")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("results/figures/test_accuracy.png", dpi=150)
plt.show()

In [ ]:
# Break down the QNN and GRU scores by the sampled noise probabilities.
group_rows = []
for model_name, pred in [
    ("lookup", lookup_predict(test["observed"])),
    ("qnn", qnn_logits),
]:
    predicted = pred.argmax(axis=-1)
    for p in sorted(np.unique(test["p"])):
        mask = test["p"] == p
        group_rows.append({"model": model_name, "group": "p", "value": float(p),
                           "accuracy": float(np.mean(np.all(predicted[mask] == y_test[mask], axis=1)))})
    for q in sorted(np.unique(test["q"])):
        mask = test["q"] == q
        group_rows.append({"model": model_name, "group": "q", "value": float(q),
                           "accuracy": float(np.mean(np.all(predicted[mask] == y_test[mask], axis=1)))})
group_df = pd.DataFrame(group_rows)
display(group_df)
group_df.to_csv("results/noise_breakdown.csv", index=False)

In [ ]:
# Cross-entropy loss for learned models, using the two 8-class outputs.
def logits_for(model, features, batch_size=64):
    model.eval()
    chunks = []
    with torch.no_grad():
        for start in range(0, len(features), batch_size):
            xb = torch.tensor(features[start:start+batch_size], dtype=torch.float32, device=device)
            chunks.append(model(xb).cpu())
    return torch.cat(chunks, dim=0)

mlp_logits = logits_for(mlp, x_mlp)
gru_logits = logits_for(gru, x_gru)
labels_t = torch.tensor(y_test, dtype=torch.long)
learned_losses = {
    "mlp": float(batch_loss(mlp_logits, labels_t).item()),
    "gru": float(batch_loss(gru_logits, labels_t).item()),
    "qnn": float(batch_loss(torch.tensor(qnn_logits), labels_t).item()),
}
training_times = {
    "mlp": float(mlp_ckpt.get("history", {}).get("training_seconds", float("nan"))),
    "gru": float(gru_ckpt.get("history", {}).get("training_seconds", float("nan"))),
    "qnn": float(qnn_ckpt.get("history", {}).get("training_seconds", float("nan"))),
}
metrics_payload = {
    "test_metrics": results,
    "cross_entropy_loss": learned_losses,
    "training_seconds": training_times,
    "qnn_inference_seconds_for_test_set": qnn_inference_seconds,
    "notes": [
        "Lookup and MLP use the latest observed syndrome; GRU uses all three rounds.",
        "Labels are reference single-qubit labels derived from the final true syndrome.",
        "QNN uses a local exact StatevectorEstimator; no quantum hardware was used.",
        "No logical-error rate is reported because full recovery/logical-observable simulation is not implemented."
    ]
}
save_json(metrics_payload, "results/metrics.json")
print("Saved results/metrics.json, results/test_comparison.csv, and figures.")

## Interpretation

Use the generated metrics rather than assuming a winner. The GRU receives more temporal information than the MLP and lookup baseline, so a higher score would not isolate architecture alone. The QNN may be slower or less accurate than classical models; that is a valid result. These experiments do not establish quantum advantage.